# Cross-Country Flights: Price Change Modeling ✈️
## Notebook 3: Feature Engineering
### Will Bartlett & Kevin Tran

This notebook imports data from Dropbox and performs feature engineering, preparing the data for modeling. The goal of these models is to predict a change in the flight's price for the next day.

### 1. Get Data From Dropbox
<b>NOTE:</b> Gets all .csv files from Dropbox, not from the sample data in the <i>data</i> folder of the repository

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import dropbox
import os
import io
from dropbox.exceptions import ApiError
from scipy import stats

In [ ]:
# Save dropbox credentials
DROPBOX_APP_KEY = "9gghuhtkyrtyn2u"
DROPBOX_APP_SECRET = "tqtvwbr15gbwseg"
DROPBOX_REFRESH_TOKEN = "LpMKV__by1wAAAAAAAAAAYn_oi2fIW0SBBqF0n6gap1WavDnJGFRMcVWQol7X5Vy"

# Create Dropbox client
dbx = dropbox.Dropbox(
    app_key=DROPBOX_APP_KEY,
    app_secret=DROPBOX_APP_SECRET,
    oauth2_refresh_token=DROPBOX_REFRESH_TOKEN
)

print("Reading files from Dropbox...")

# List all files
result = dbx.files_list_folder('/flight_data')

dfs = []
for entry in result.entries:
    if entry.name.endswith('.csv'):
        print(f"Reading: {entry.name}")
        
        # Download file content
        metadata, response = dbx.files_download(entry.path_lower)
        
        # Read directly into pandas
        df = pd.read_csv(io.BytesIO(response.content))
        dfs.append(df)

# Combine
combined_df = pd.concat(dfs, ignore_index=True)

# Process
combined_df['route'] = combined_df['origin'] + ' → ' + combined_df['destination']
combined_df['time_collected'] = pd.to_datetime(combined_df['time_collected'])
combined_df['departure_date'] = pd.to_datetime(combined_df['departure_date'])
df = combined_df.copy()

# Convert to datetime with error handling
try:
    if df['departure_date'].dtype != 'datetime64[ns]':
        df['departure_date'] = pd.to_datetime(df['departure_date'], errors='coerce')
except Exception as e:
    print(f"Error with departure_date: {e}")
    # Try alternative parsing
    df['departure_date'] = pd.to_datetime(df['departure_date'].astype(str), errors='coerce')

try:
    if df['departure_time'].dtype != 'datetime64[ns]':
        df['departure_time'] = pd.to_datetime(df['departure_time'], errors='coerce')
except Exception as e:
    print(f"Error with departure_time: {e}")
    df['departure_time'] = pd.to_datetime(df['departure_time'].astype(str), errors='coerce')

try:
    if df['time_collected'].dtype != 'datetime64[ns]':
        df['time_collected'] = pd.to_datetime(df['time_collected'], errors='coerce')
except Exception as e:
    print(f"Error with time_collected: {e}")
    df['time_collected'] = pd.to_datetime(df['time_collected'].astype(str), errors='coerce')

# Round departure time to nearest 15 minutes (flights might vary by a few minutes)
df['departure_time_rounded'] = df['departure_time'].dt.floor('15min')

# Create flight ID for each unique flight
df['flight_id'] = (
    df['origin'].astype(str) + '_' + 
    df['destination'].astype(str) + '_' + 
    df['departure_date'].dt.strftime('%Y%m%d') + '_' + 
    df['departure_time_rounded'].dt.strftime('%H%M') + '_' + 
    df['airline'].astype(str)
)
df.head()

### 2. X Feature

### 3. Y Feature

In [ ]:
df_engineered = df 

In [ ]:
# Save the engineered data; ignored in repository
df_engineered.to_csv('data/features_engineered.csv', index=False)

### 